# Module 07 — Heaps Priority Queues and TopK Patterns

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

import pytest   # some assertions check that an invalid input RAISES
sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_kth_largest import kth_largest
from p02_merge_k_sorted import merge_k_sorted
from p03_streaming_median import streaming_median

print("module 07: Heaps Priority Queues and TopK Patterns")
print("problems available:", 8)
for name in ['p01_kth_largest', 'p02_merge_k_sorted', 'p03_streaming_median', 'p04_k_closest_points', 'p05_last_stone_weight', 'p06_min_meeting_rooms', 'p07_task_scheduler', 'p08_reorganize_string']:
    print(f"  {name}")

## 1. Baseline — `p01_kth_largest`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
assert kth_largest([1], 1) == 1
# k = 1 is the maximum, k = n is the minimum.
assert kth_largest([7, 3, 9, 1], 1) == 9
assert kth_largest([7, 3, 9, 1], 4) == 1
# Duplicates occupy distinct positions.
assert kth_largest([5, 5, 5], 2) == 5
assert kth_largest([2, 2, 1], 2) == 2
# Negative values.
assert kth_largest([-1, -5, -3], 2) == -3
with pytest.raises(ValueError):
    kth_largest([1, 2], 3)
with pytest.raises(ValueError):
    kth_largest([1, 2], 0)
# Cross-check every k against a sort.
data = [9, 4, 7, 1, 7, 3, 8, 2]
ordered = sorted(data, reverse=True)
for k in range(1, len(data) + 1):
    assert kth_largest(data, k) == ordered[k - 1], k
# Scale.
big = list(range(100_000))
assert kth_largest(big, 3) == 99_997

print("all assertions held")

## 2. Predict before you run

You want the 3rd largest of a million numbers. Predict which heap you need and what its maximum size should be. Most people answer 'max-heap' and it is the wrong one — work out why before running the cell.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert merge_k_sorted([[1, 4, 5], [1, 3, 4], [2, 6]]) == [1, 1, 2, 3, 4, 4, 5, 6]
assert merge_k_sorted([]) == []
assert merge_k_sorted([[]]) == []
assert merge_k_sorted([[], []]) == []
assert merge_k_sorted([[1]]) == [1]
# Some lists empty, some not.
assert merge_k_sorted([[], [1, 2], []]) == [1, 2]
# Disjoint ranges.
assert merge_k_sorted([[1, 2], [3, 4], [5, 6]]) == [1, 2, 3, 4, 5, 6]
assert merge_k_sorted([[5, 6], [3, 4], [1, 2]]) == [1, 2, 3, 4, 5, 6]
# All values identical across lists - the tie-breaking case.
assert merge_k_sorted([[2, 2], [2], [2, 2]]) == [2] * 5
# Negative values.
assert merge_k_sorted([[-3, -1], [-2, 0]]) == [-3, -2, -1, 0]
# Cross-check against concatenate-and-sort.
data = [[1, 5, 9], [2, 2, 8], [], [0, 3, 3, 7], [4]]
assert merge_k_sorted(data) == sorted(x for lst in data for x in lst)
# Scale: 1000 lists of 100 elements.
many = [list(range(i, i + 100)) for i in range(1000)]
merged = merge_k_sorted(many)
assert len(merged) == 100_000 and merged == sorted(merged)

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert streaming_median([2, 3, 4]) == [2.0, 2.5, 3.0]
assert streaming_median([1]) == [1.0]
assert streaming_median([1, 2]) == [1.0, 1.5]
# Descending input exercises the rebalancing hardest.
assert streaming_median([5, 4, 3, 2, 1]) == [5.0, 4.5, 4.0, 3.5, 3.0]
# Duplicates.
assert streaming_median([2, 2, 2]) == [2.0, 2.0, 2.0]
# Negatives.
assert streaming_median([-1, -2, -3]) == [-1.0, -1.5, -2.0]
# Cross-check against a naive re-sort at every step.
import statistics
data = [41, 35, 62, 5, 97, 97, 21, 3, 88, 54, 11]
expected = [
    float(statistics.median(data[: i + 1])) for i in range(len(data))
]
assert streaming_median(data) == expected
# Scale: the naive approach would be ~10^10 operations.
big = list(range(50_000))
med = streaming_median(big)
assert len(med) == 50_000
# Median of 0..49999 is the mean of the two middle values.
assert med[-1] == (24_999 + 25_000) / 2
assert med[0] == 0.0

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. To keep the k largest, use a MIN-heap of size k. The root is what you evict.
2. Two heaps give a streaming median, and the size invariant is the algorithm.
3. A heap is the wrong tool when counting or bucketing gets you O(n).

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem